# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/M-Sheheryar-khan/FlyRank-ML-Internship-Starter-Repo/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb
import os
import duckdb
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

In [2]:
features = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions_h1,
        SUM(gsc_clicks) AS clicks_h1,
        AVG(gsc_avg_position) AS avg_position_h1,
        COUNT(DISTINCT report_date) AS active_days_h1
    FROM {MAR}
    WHERE report_date <= DATE '2026-03-15'
    GROUP BY content_hash_id
    HAVING impressions_h1 >= 20
""").df()

features["ctr_h1"] = features["clicks_h1"] / features["impressions_h1"] * 100

print(features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(109592, 6)


,content_hash_id,impressions_h1,clicks_h1,avg_position_h1,active_days_h1,ctr_h1
0,content_39d7361b4945d504,57.0,0.0,3.659683,15,0.000000
1,content_cec711b02f3bbde6,199.0,2.0,4.086084,15,1.005025
2,content_275b6f7f733016d4,467.0,1.0,4.449176,15,0.214133
3,content_ceaec531566ffcfc,56.0,0.0,6.600595,15,0.000000
4,content_755d951187fcd70a,771.0,1.0,1.883472,15,0.129702


In [3]:
labels = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions_h2
    FROM {MAR}
    WHERE report_date > DATE '2026-03-15'
    GROUP BY content_hash_id
""").df()

data = features.merge(labels, on="content_hash_id", how="left")
data["impressions_h2"] = data["impressions_h2"].fillna(0)

data["is_declining"] = (data["impressions_h2"] < 0.8 * data["impressions_h1"]).astype(int)

print("declining rate:", data["is_declining"].mean().round(3))
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

declining rate: 0.291


,content_hash_id,impressions_h1,clicks_h1,avg_position_h1,active_days_h1,ctr_h1,impressions_h2,is_declining
0,content_39d7361b4945d504,57.0,0.0,3.659683,15,0.000000,20.0,1
1,content_cec711b02f3bbde6,199.0,2.0,4.086084,15,1.005025,403.0,0
2,content_275b6f7f733016d4,467.0,1.0,4.449176,15,0.214133,343.0,1
3,content_ceaec531566ffcfc,56.0,0.0,6.600595,15,0.000000,26.0,1
4,content_755d951187fcd70a,771.0,1.0,1.883472,15,0.129702,1087.0,0


In [4]:
dim_content = con.sql(f"""
    SELECT content_hash_id, content_type, content_created_date, content_updated_date
    FROM read_parquet('{REL}/dim_content.parquet')
""").df()

print(dim_content.shape)
dim_content.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(519606, 4)


,content_hash_id,content_type,content_created_date,content_updated_date
0,content_004de9653278b5a4,keyword article,2026-05-30,2026-07-01
1,content_00dc5efae381b2ab,keyword article,2026-06-12,2026-07-01
2,content_01410f2556c327ac,keyword article,2026-05-09,2026-07-01
3,content_019f27f634053ca7,keyword article,2026-06-15,2026-06-15
4,content_01efa71faea45dcc,keyword article,2026-05-21,2026-06-01


In [5]:
client_lookup = con.sql(f"""
    SELECT DISTINCT content_hash_id, client_hash_id
    FROM {MAR}
""").df()

data = data.merge(client_lookup, on="content_hash_id", how="left")
data.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(109592, 9)

## 1. My rule and its reason codes

Signal 1 (staleness) verdict: FALSE. 99% of content in this March slice (20,239 of 20,441 rows) was updated within the last 90 days — barely anything is actually stale here. Decline rate doesn't rise with staleness (37% → 33% → 38%), and the 180-365 bucket only has 26 rows, too small to trust. Staleness isn't a usable signal in this data — dropping it from the rule.

In [6]:
# check dim_content actually has it
print(dim_content.columns.tolist())

# merge it onto data
data = data.merge(dim_content[["content_hash_id", "content_updated_date"]],
                   on="content_hash_id", how="left")

print(data.columns.tolist())

['content_hash_id', 'content_type', 'content_created_date', 'content_updated_date']
['content_hash_id', 'impressions_h1', 'clicks_h1', 'avg_position_h1', 'active_days_h1', 'ctr_h1', 'impressions_h2', 'is_declining', 'client_hash_id', 'content_updated_date']


In [8]:
import pandas as pd
data["content_updated_date"] = pd.to_datetime(data["content_updated_date"])

data["days_since_update"] = (pd.Timestamp("2026-03-15") - data["content_updated_date"]).dt.days
data["staleness_tier"] = pd.cut(data["days_since_update"], bins=[0,90,180,365,99999],
                                 labels=["0-90","90-180","180-365","365+"])

bucket1 = data.groupby("staleness_tier").agg(
    n=("content_hash_id", "size"),
    decline_rate=("is_declining", "mean")
).round(3)
print(bucket1)

                    n  decline_rate
staleness_tier                     
0-90            20239         0.371
90-180            176         0.335
180-365            26         0.385
365+                0           NaN


/tmp/ipykernel_3267/3355533138.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket1 = data.groupby("staleness_tier").agg(


In [9]:
data["position_tier"] = pd.cut(data["avg_position_h1"], bins=[0,3,10,20,9999],
                                labels=["1-3","4-10","11-20","21+"])

bucket2 = data.groupby("position_tier", observed=True).apply(
    lambda g: pd.Series({
        "n": len(g),
        "weighted_ctr": g["clicks_h1"].sum() / g["impressions_h1"].sum() * 100
    })
).round(3)
print(bucket2)

                     n  weighted_ctr
position_tier                       
1-3            11583.0         0.449
4-10           49444.0         0.328
11-20          21597.0         0.337
21+            26958.0         0.135


/tmp/ipykernel_3267/3052767300.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  bucket2 = data.groupby("position_tier", observed=True).apply(


Signal 2 (CTR vs. position) verdict: CONFIRMED. Weighted CTR falls as position gets worse — 0.449% in positions 1-3, down to 0.328–0.337% in the 4-20 range, then a clear drop to 0.135% at 21+ (n = 11,583 / 49,444 / 21,597 / 26,958). The middle two tiers are close to flat, but the top tier and the bottom tier are clearly separated, which is the direction FlyRank's needs_ctr_fix logic assumes. This signal is usable for the rule — position tier gives a fair "expected CTR" baseline to compare a page against.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [11]:
import numpy as np
import os

data["expected_ctr"] = data["position_tier"].map(bucket2["weighted_ctr"].to_dict()).astype(float)
data["ctr_gap"] = data["expected_ctr"] - data["ctr_h1"]

VISIBLE_MIN_IMPRESSIONS = 100

is_underperform = (data["ctr_gap"] > 0) & (data["impressions_h1"] >= VISIBLE_MIN_IMPRESSIONS)

data["baseline_score"] = np.where(is_underperform, data["impressions_h1"] * data["ctr_gap"], 0)
data["reason_code"] = np.where(is_underperform, "ctr_underperform_visible", "no_flag")
data["action"] = np.where(is_underperform, "review_ctr", "monitor")

os.makedirs("work/outputs", exist_ok=True)
queue = data.sort_values("baseline_score", ascending=False).reset_index(drop=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Rows written:", len(queue))
print("Flagged for review:", (queue["action"] == "review_ctr").sum())
queue[["content_hash_id", "position_tier", "impressions_h1", "avg_position_h1",
       "ctr_h1", "expected_ctr", "ctr_gap", "baseline_score", "reason_code", "action"]].head(10)

Rows written: 109592
Flagged for review: 52258


,content_hash_id,position_tier,impressions_h1,avg_position_h1,ctr_h1,expected_ctr,ctr_gap,baseline_score,reason_code,action
0,content_34a70fea29d15f24,1-3,73639.0,2.786744,0.024444,0.449,0.424556,31263.911,ctr_underperform_visible,review_ctr
1,content_9c057b66c30a3abb,4-10,83772.0,8.607910,0.000000,0.328,0.328000,27477.216,ctr_underperform_visible,review_ctr
2,content_7c6373141eae744a,4-10,86860.0,5.785512,0.058715,0.328,0.269285,23390.080,ctr_underperform_visible,review_ctr
3,content_82e35c4845e6c391,11-20,70169.0,18.269589,0.041329,0.337,0.295671,20746.953,ctr_underperform_visible,review_ctr
4,content_8e1334d6356668e3,4-10,58553.0,4.579049,0.001708,0.328,0.326292,19105.384,ctr_underperform_visible,review_ctr
5,content_65c75874a23fca87,4-10,55680.0,9.013531,0.026940,0.328,0.301060,16763.040,ctr_underperform_visible,review_ctr
6,content_945d6ff91386c817,4-10,49314.0,6.413782,0.004056,0.328,0.323944,15974.992,ctr_underperform_visible,review_ctr
7,content_f6116743b00afc2d,4-10,49619.0,9.493028,0.016123,0.328,0.311877,15475.032,ctr_underperform_visible,review_ctr
8,content_1642f339bd6e7c8d,4-10,52378.0,4.011050,0.034366,0.328,0.293634,15379.984,ctr_underperform_visible,review_ctr
9,content_acbcc847f8996314,4-10,83715.0,3.453361,0.158872,0.328,0.169128,14158.520,ctr_underperform_visible,review_ctr


1. content_34a70fea29d15f24 — review_ctr — 73,639 impressions at position 2.8 (top of page 1) but CTR only 0.024% vs an expected 0.449% for that tier — would be wrong if the query is informational/branded and genuinely doesn't drive clicks even at position 1-3 (e.g. a "what is X" query where users read the snippet and don't click through).

2. content_9c057b66c30a3abb — review_ctr — 83,772 impressions at position 8.6 with CTR literally 0.000% vs an expected 0.328% — would be wrong if the click tracking itself is broken or misattributed for this page rather than the page underperforming.

3. content_7c6373141eae744a — review_ctr — 86,860 impressions at position 5.8, CTR 0.059% vs expected 0.328% — would be wrong if the title/meta description is fine and the real issue is a SERP feature (featured snippet, PAA box) siphoning clicks before users reach this result.

4. content_82e35c4845e6c391 — review_ctr — 70,169 impressions at position 18.3, CTR 0.041% vs expected 0.337% — would be wrong if position 18 rarely gets seen at all regardless of title quality, making this a ranking problem rather than a CTR problem.

5. content_8e1334d6356668e3 — review_ctr — 58,553 impressions at position 4.6, CTR 0.002% vs expected 0.328% — would be wrong if this page recently changed URL/redirected and GSC is still attributing old click data inconsistently with new impression data.

6. content_65c75874a23fca87 — review_ctr — 55,680 impressions at position 9.0, CTR 0.027% vs expected 0.328% — would be wrong if the query intent is navigational (users already know the brand/URL) and click through search rarely, regardless of snippet quality.

7. content_945d6ff91386c817 — review_ctr — 49,314 impressions at position 6.4, CTR 0.004% vs expected 0.328% — would be wrong if this is a duplicate/near-duplicate of a stronger page cannibalizing its own clicks.

8. content_f6116743b00afc2d — review_ctr — 49,619 impressions at position 9.5, CTR 0.016% vs expected 0.328% — would be wrong if the position is unstable (bouncing between page 1 and page 2) and the 15-day average masks that instability as a CTR issue.

9. content_1642f339bd6e7c8d — review_ctr — 52,378 impressions at position 4.0, CTR 0.034% vs expected 0.328% — would be wrong if the ranking keyword itself has low commercial/click intent (e.g. a broad head term) compared to the tier's average.

10. content_acbcc847f8996314 — review_ctr — 83,715 impressions at position 3.5, CTR 0.159% vs expected 0.328% — the smallest gap in the top 10, so this is closer to a borderline call — would be wrong if 0.159% is actually normal for this page's content type and the tier-wide 0.449% average is being pulled up by a few outlier pages.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [12]:
top10 = queue.head(10)[["content_hash_id", "position_tier", "impressions_h1",
                          "avg_position_h1", "ctr_h1", "expected_ctr", "ctr_gap",
                          "baseline_score", "reason_code", "action"]]
top10

,content_hash_id,position_tier,impressions_h1,avg_position_h1,ctr_h1,expected_ctr,ctr_gap,baseline_score,reason_code,action
0,content_34a70fea29d15f24,1-3,73639.0,2.786744,0.024444,0.449,0.424556,31263.911,ctr_underperform_visible,review_ctr
1,content_9c057b66c30a3abb,4-10,83772.0,8.607910,0.000000,0.328,0.328000,27477.216,ctr_underperform_visible,review_ctr
2,content_7c6373141eae744a,4-10,86860.0,5.785512,0.058715,0.328,0.269285,23390.080,ctr_underperform_visible,review_ctr
3,content_82e35c4845e6c391,11-20,70169.0,18.269589,0.041329,0.337,0.295671,20746.953,ctr_underperform_visible,review_ctr
4,content_8e1334d6356668e3,4-10,58553.0,4.579049,0.001708,0.328,0.326292,19105.384,ctr_underperform_visible,review_ctr
5,content_65c75874a23fca87,4-10,55680.0,9.013531,0.026940,0.328,0.301060,16763.040,ctr_underperform_visible,review_ctr
6,content_945d6ff91386c817,4-10,49314.0,6.413782,0.004056,0.328,0.323944,15974.992,ctr_underperform_visible,review_ctr
7,content_f6116743b00afc2d,4-10,49619.0,9.493028,0.016123,0.328,0.311877,15475.032,ctr_underperform_visible,review_ctr
8,content_1642f339bd6e7c8d,4-10,52378.0,4.011050,0.034366,0.328,0.293634,15379.984,ctr_underperform_visible,review_ctr
9,content_acbcc847f8996314,4-10,83715.0,3.453361,0.158872,0.328,0.169128,14158.520,ctr_underperform_visible,review_ctr


1. content_XXXX — review_ctr — [impressions] impressions at position [X] but CTR [Y]% vs expected [Z]% for that tier — would be wrong if this page targets a low-click intent (e.g. a definitional query) rather than a genuinely underperforming one.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [13]:
weak = queue[queue["action"] == "review_ctr"].tail(5)[
    ["content_hash_id", "position_tier", "impressions_h1", "ctr_h1", "expected_ctr", "baseline_score"]
]
print(weak)

used_in_score = ["impressions_h1", "position_tier", "expected_ctr", "ctr_gap"]
future_or_label_cols = ["impressions_h2", "is_declining"]

print("Columns used in baseline_score:", used_in_score)
assert not any(c in used_in_score for c in future_or_label_cols), "leakage: future-window column used in score"
print("No future-window column (impressions_h2, is_declining) went into the score.")
print("FlyRank product flags (health_score, priority_score, action_type, refresh flags) are not in this dataset — nothing to accidentally leak.")

                content_hash_id position_tier  impressions_h1    ctr_h1  \
52253  content_656d174f68095ac7          4-10           305.0  0.327869   
52254  content_0e810a74c734e7b5          4-10           305.0  0.327869   
52255  content_6b7b408125a89a76          4-10           305.0  0.327869   
52256  content_5926da62238d6f8d           21+           741.0  0.134953   
52257  content_89cf2bfb1ca7fbc5           21+           741.0  0.134953   

       expected_ctr  baseline_score  
52253         0.328           0.040  
52254         0.328           0.040  
52255         0.328           0.040  
52256         0.135           0.035  
52257         0.135           0.035  
Columns used in baseline_score: ['impressions_h1', 'position_tier', 'expected_ctr', 'ctr_gap']
No future-window column (impressions_h2, is_declining) went into the score.
FlyRank product flags (health_score, priority_score, action_type, refresh flags) are not in this dataset — nothing to accidentally leak.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.